[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-Music3_Colab.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-Music3_Colab.ipynb)

# 🎵 MiniMax-Music3 — Lyrics + Caption-to-Music (Qwen3 + Flow-Matching DiT + Flow-VAE)
> **Runtime required:** GPU. **Recommended:** **L4 (24 GB)** or **A100 (40/80 GB)**. T4 (16 GB) works with CPU offloading enabled (slower). 8 GB GPUs work with group-offloading (much slower, ~10-20x).
A Colab port of [MiniMax-Music3](https://huggingface.co/MiniMaxAI/MiniMax-Music3) — a lyrics- and caption-conditioned music generation model that produces **stereo audio at 44.1 kHz** with vocals, instruments, and structure. Built on the official [diffusers `MiniMaxMusic3ModularPipeline`](https://github.com/huggingface/diffusers/tree/main/src/diffusers/modular_pipelines/minimax_music3) (merged into `main` 2026-08-13).
## How it works
MiniMax-Music3 is a **three-stage pipeline**:
1. **Semantic generation** — Qwen3-8B language model + RVQ depth decoder reads the structured caption (genre / mood / vocals / instrumentation / arrangement) + tagged lyrics (`[verse]`, `[chorus]`, `[instrumental]`, ...) and autoregressively generates per-frame semantic tokens + hidden states (25 Hz AR frame rate).
2. **Core denoise** — 2.4B flow-matching transformer runs 30 Euler steps per 200-frame window (with 100-frame overlap) to generate the 128-channel Flow-VAE audio latent from noise, conditioned on the LM's hidden states.
3. **Vocoder decode** — 123M Flow-VAE vocoder turns the audio latents into a stereo waveform at **44.1 kHz**. Windows are stitched via overlap-add (drops leading 86 / trailing 344-86 latent frames per window).
```
prompt + lyrics -> Qwen3-8B + RVQ depth decoder -> per-frame hidden states
                                          |
                                          v
                          Flow-Matching DiT (2.4B, 30 steps/window)
                                          |
                                          v
                                  Flow-VAE latent (128-ch)
                                          |
                                          v
                                Flow-VAE vocoder (123M)
                                          |
                                          v
                                  stereo WAV @ 44.1 kHz
```
## ⚠️ License
Released under the **MiniMax Music 3 Community License**. Check the model card before commercial use.
## Quick start
1. **Runtime → Change runtime type → GPU** (L4, A100, or A100 80GB)
2. Run **STEP 1** — installs torch 2.11.0+cu128, diffusers from `main` (MiniMax-Music3 is in `main` as of 2026-08-13), torchaudio, scipy. First run: ~5-10 min.
3. Run **STEP 2** — downloads the 7-component modular checkpoint (~27 GB total: 16 GB Qwen3-8B + 9 GB DiT + 1.2 GB RVQ decoder + 0.5 GB misc). First run: ~15-30 min.
4. Run **STEP 3** — imports + lazy pipe loader. Choose CPU offloading mode for T4 (16 GB) or 8 GB GPUs.
5. Run **STEP 4** for the Gradio UI, or **STEP 6** for a single-song quick test, or **STEP 7** for batch.
6. **STEP 5** keep-alive prevents Colab from disconnecting.
7. **STEP 9** (optional) muxes the generated song with an mp4 from your LTX-2-5 video gen, producing a final music-video output.## MemoryThe model is bf16-only - there's no INT8/INT4 quantized release, but the diffusers `ModularPipeline` supports CPU offloading via `ComponentsManager` and leaf-level group offloading (added in PR #14456). Three modes are exposed via the `OFFLOAD_MODE` form widget:
| GPU | VRAM | Mode | Peak VRAM | 60s song runtime | Notes ||-----|------|------|-----------|------------------|-------|| **A100 80GB** | 80 GB | `bf16` | ~26 GB | ~2-3 min | Full quality, headroom for long clips |
| **A100 40GB** | 40 GB | `bf16` | ~26 GB | ~2-3 min | Full quality, fits comfortably |
| **L4 24GB** | 24 GB | `bf16` | ~24 GB | ~5-7 min | Tight but works. Expect OOM risk on very long clips. |
| **T4 16GB** | 16 GB | `cpu_offload` | ~16 GB peak | ~15-20 min | Works with `enable_auto_cpu_offload` on the ComponentsManager. ~3-4x slower. |
Total checkpoint size on disk: ~27 GB (16 GB Qwen3-8B + 9 GB DiT + 1.2 GB RVQ decoder + 0.5 GB misc).If you're on T4: enable `OFFLOAD_MODE = 'cpu_offload'` in STEP 3. The 8B Qwen3 LM will be CPU-resident except during the AR stage; the 2.4B DiT, vocoder, and condition encoder stay on GPU. Expect ~3-4x slower than full GPU but it fits in 16 GB. For 8 GB GPUs, additionally use `group_offload` to stream the LM weights one layer at a time (~10-20x slower).
## Outputs
Each generation produces a **stereo WAV file** at 44.1 kHz (`*.wav` in `AEI_3D_Out/MiniMax-Music3/songs/`). The file is **float32 in `[-1, 1]`** written as **PCM int16** for compatibility with all audio editors.
Default song length is 60 seconds. Max is 360 seconds (6 min, capped at 9000 acoustic frames).

In [ ]:
#@title STEP 1 — Install torch, diffusers main, torchaudio, scipy
"""
• Pins torch to 2.11.0+cu128 (recent enough for the MiniMax-Music3 modular pipeline)
• Installs diffusers from main (MiniMax-Music3 is merged as of 2026-08-13, no PR pin needed)
• Installs torchaudio (for audio utilities)
• Installs scipy (for WAV writing via scipy.io.wavfile.write)
• Stubs the `spaces` module (HF ZeroGPU API, not available on Colab)
• Mounts Google Drive for checkpoint caching
"""
import os, sys, time, subprocess, pathlib
print('='*72)
print('MiniMax-Music3 - Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()
CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MiniMax-Music3')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_mm_music3_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')
OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-Music3')
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
# --- 1. Install / upgrade torch ---
t0 = time.time()
print('\n[1/4] Ensuring torch 2.11.0+cu128 ...')
try:
    import torch
    if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 11):
        print(f'  Upgrading torch {torch.__version__} -> 2.11.0+cu128 ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                        'torch>=2.11.0,<2.12.0',
                        '--index-url', 'https://download.pytorch.org/whl/cu128'],
                       check=True)
    else:
        print(f'  torch {torch.__version__} OK')
except ImportError:
    print('  Installing torch 2.11.0+cu128 ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch>=2.11.0,<2.12.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu128'],
                   check=True)
print(f'  torch ready in {time.time()-t0:.1f}s')
# --- 2. Install diffusers from main (MiniMax-Music3 is in MODULAR_PIPELINE_MAPPING) ---
t0 = time.time()
print('\n[2/4] Installing diffusers from main ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'git+https://github.com/huggingface/diffusers.git',
                '--upgrade'],
               check=True)
print(f'  diffusers installed in {time.time()-t0:.1f}s')
# --- 3. Install supporting deps ---
t0 = time.time()
print('\n[3/4] Installing supporting deps (torchaudio, scipy, soundfile) ...')
EXTRA_PKGS = [
    'torchaudio',
    'scipy',
    'soundfile',
    'librosa',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
print(f'  extra deps ready in {time.time()-t0:.1f}s')
# --- 4. Stub the `spaces` module (HF ZeroGPU API, not available on Colab) ---
t0 = time.time()
print('\n[4/4] Stubbing `spaces` module ...')
import types as _types
if 'spaces' not in sys.modules:
    _spaces_stub = _types.ModuleType('spaces')
    def _noop_decorator(*args, **kwargs):
        # Handle both @spaces.GPU and @spaces.GPU(duration=...) forms
        if len(args) == 1 and callable(args[0]) and not kwargs:
            return args[0]
        def _decorator(fn):
            return fn
        return _decorator
    _spaces_stub.GPU = _noop_decorator
    _spaces_stub.CUDA = _noop_decorator
    _spaces_stub.aoti_load_from_package_dir = lambda *a, **kw: None
    sys.modules['spaces'] = _spaces_stub
    print('  spaces stub registered (no-op)')
else:
    print('  spaces already loaded (likely from previous run)')
print(f'  stub setup in {time.time()-t0:.1f}s')
# --- Verify imports ---
print('\nVerifying imports ...')
try:
    import diffusers
    print(f'  diffusers     : {diffusers.__version__}')
except Exception as e:
    print(f'  [FAIL] diffusers: {e}')
    raise
try:
    from diffusers import ModularPipeline
    print('  ModularPipeline: importable')
except Exception as e:
    print(f'  [FAIL] ModularPipeline: {e}')
    raise
try:
    from diffusers.modular_pipelines.minimax_music3 import MiniMaxMusic3ModularPipeline
    print('  MiniMaxMusic3ModularPipeline: importable')
except Exception as e:
    print(f'  [FAIL] MiniMaxMusic3ModularPipeline: {e}')
    raise
try:
    from diffusers.models import MiniMaxMusic3Transformer1DModel
    print('  MiniMaxMusic3Transformer1DModel: importable')
except Exception as e:
    print(f'  [WARN] model class: {e}')
try:
    import torchaudio, scipy.io.wavfile, soundfile, librosa
    print('  torchaudio, scipy, soundfile, librosa: OK')
except Exception as e:
    print(f'  [FAIL] audio deps: {e}')
    raise
print('\n' + '='*72)
print('STEP 1 done. Next: run STEP 2 to download weights (~27 GB).')
print('='*72)

In [ ]:
#@title STEP 2 — Download MiniMax-Music3 weights to Drive cache
"""
Downloads the 7-component modular checkpoint via `snapshot_download`.
Total: ~27 GB on Drive. Cached; subsequent runs reuse the cache.
Components (per the modular_model_index.json):
  - tokenizer/           (Qwen2Tokenizer JSON files, ~10 MB)
  - language_model/      (Qwen3-8B, ~16 GB total, 4 shards)
  - condition_encoder/   (MiniMaxMusic3ConditionEncoder, ~96 MB)
  - rvq_depth_decoder/   (MiniMaxMusic3RVQDepthDecoder, ~1.2 GB)
  - transformer/         (MiniMaxMusic3Transformer1DModel, ~9 GB, 2 shards)
  - vocoder/             (MiniMaxMusic3Vocoder, ~200 MB)
  - scheduler/           (FlowMatchEulerDiscreteScheduler, tiny JSON)
The top-level dav.pth (~500 MB) and flowmatching_vae.pth (~10 GB) are NOT
downloaded - those are for the legacy SGLang/transformers-based loader, not
the diffusers ModularPipeline we use here. The diffusers loader only needs
the 7 modular components above.
First run: ~15-30 min depending on Drive FUSE throughput. Subsequent: 0 sec
(cached). The cache lives at `<HF_HOME>/hub/models--MiniMaxAI--MiniMax-Music3/`.
Public repo, no HF token required (unlike the gated LTX-Video-2.5 repo).
"""
import os, sys, time, pathlib
from huggingface_hub import snapshot_download
# Same cache root as STEP 1
cache_root_str = os.environ.get('HF_HOME', '/content/drive/MyDrive/AEI_3D_Cache/MiniMax-Music3/huggingface')
cache_root = pathlib.Path(cache_root_str)
OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-Music3')
REPO_ID = 'MiniMaxAI/MiniMax-Music3'
# Diffusers modular components. The top-level dav.pth + flowmatching_vae.pth
# are for the SGLang/transformers-based loader, not diffusers - skip them.
ALLOW_PATTERNS = [
    # Modular pipeline index (defines the 7 components)
    'modular_model_index.json',
    'config.json',
    # Tokenizer (Qwen2Tokenizer)
    'tokenizer/*.json',
    'tokenizer/*.jinja',
    # Language model (Qwen3-8B, 4 shards + index)
    'language_model/*.safetensors',
    'language_model/*.index.json',
    'language_model/*.json',
    # Condition encoder
    'condition_encoder/*.safetensors',
    'condition_encoder/*.json',
    # RVQ depth decoder
    'rvq_depth_decoder/*.safetensors',
    'rvq_depth_decoder/*.json',
    # Transformer (2.4B DiT, 2 shards + index)
    'transformer/*.safetensors',
    'transformer/*.index.json',
    'transformer/*.json',
    # Vocoder (Flow-VAE)
    'vocoder/*.safetensors',
    'vocoder/*.json',
    # Scheduler (FlowMatchEulerDiscreteScheduler)
    'scheduler/*.json',
    # Docs (for the markdown intro)
    'README.md',
    'LICENSE',
]
print('='*72)
print('Downloading ' + REPO_ID + ' to ' + str(cache_root) + '/hub/ ...')
print('Allow patterns: ' + str(len(ALLOW_PATTERNS)) + ' entries')
print('Public repo - no auth required.')
print('='*72)
t0 = time.time()
try:
    local_path = snapshot_download(
        repo_id=REPO_ID,
        cache_dir=str(cache_root),
        allow_patterns=ALLOW_PATTERNS,
        max_workers=4,
    )
    elapsed = time.time() - t0
    print('\nDone in ' + str(round(elapsed/60, 1)) + ' min. Local path: ' + local_path)
    # Compute total size
    total_bytes = 0
    for root, dirs, files in os.walk(local_path):
        for f in files:
            fp = os.path.join(root, f)
            total_bytes += os.path.getsize(fp)
    print('Total on disk: ' + str(round(total_bytes / 1024**3, 2)) + ' GB')
    # Show per-component size
    print('\nComponents present:')
    for entry in sorted(os.listdir(local_path)):
        full = os.path.join(local_path, entry)
        if os.path.isdir(full):
            sz = sum(os.path.getsize(os.path.join(r, f))
                    for r, _, files in os.walk(full)
                    for f in files)
            print('  ' + entry.ljust(30) + ' ' + format(sz/1024**3, '6.2f') + ' GB')
        else:
            print('  ' + entry.ljust(30) + ' (file, ' + format(os.path.getsize(full)/1024, '.1f') + ' KB)')
    print('\n' + '='*72)
    print('STEP 2 done. Next: run STEP 3 to load the pipe.')
    print('='*72)
except Exception as e:
    print('\n[FAIL] download error: ' + type(e).__name__ + ': ' + str(e))
    print('\nPossible causes:')
    print('  1. No internet connection')
    print('  2. ' + REPO_ID + ' is unreachable (network issue on Colab side - retry once or twice)')
    print('  3. Drive not mounted (set CONNECT_GOOGLE_DRIVE=True in STEP 1)')
    print('  4. Colab session timed out (re-run STEP 1 to reconnect Drive, then STEP 2)')
    raise

In [ ]:
#@title STEP 3 — Imports, lazy pipe loader, generate_song()
"""
Imports + a single `generate_song()` function that wraps the pipeline call.
The pipe is loaded once on first call and cached in a module-level global.
Subsequent calls reuse it. Lazy loading keeps cell execution fast when the
user is just tweaking prompts or JSON.
Three offload modes (form widget OFFLOAD_MODE):
  - 'none'        - everything on GPU (~24 GB; needs L4 or A100)
  - 'cpu_offload'  - paged CPU offload (works on T4 16 GB, ~3-4x slower)
  - 'group_offload' - CPU offload + LM leaf-level streaming (works on 8 GB, ~10-20x slower)
Tuning tip: if you OOM on a long song, the AR stage (Qwen3-8B) is the
main VRAM consumer; switching to 'cpu_offload' drops peak by ~8 GB.
Output: stereo WAV file at 44.1 kHz (PCM int16). The official README mentions
32 kHz, but the diffusers pipeline vocoder outputs at 44.1 kHz (the vocoder
config's sampling_rate property; see `pipe.sampling_rate` after loading).
What is exposed to other cells via builtins:
  - _generate_song(prompt, lyrics, audio_duration, num_inference_steps, seed, ...)
  - _load_pipe()         (re-runs the pipe loader, e.g. after a mode change)
  - _save_wav(...)       (WAV writer for manual saving)
  - MiniMax_MUSIC3_OUT_DIR, MiniMax_MUSIC3_REPO_DIR (paths)
"""
import os, sys, time, json, gc, random, builtins, pathlib
from pathlib import Path
from typing import Optional
import numpy as np
import torch
import scipy.io.wavfile as scipy_wav
URL = 'http://127.0.0.1:8188'  # Not used (no ComfyUI), kept for AEI-compat
_cache_root_str = os.environ.get('HF_HOME', '/content/drive/MyDrive/AEI_3D_Cache/MiniMax-Music3/huggingface')
OUT_DIR = Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-Music3')
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Lazy global pipe state
PIPE = None
PIPE_MODE = None
REPO_DIR = None
# ---- form widget for mode ----
OFFLOAD_MODE = 'none'  #@param ['none', 'cpu_offload', 'group_offload']
# Where to host model components to fit smaller VRAM (see choices below).
#   'none'         - everything on GPU (~24 GB; needs L4 or A100).
#   'cpu_offload'  - models live on CPU, paged to GPU per forward pass (~16 GB; works on T4).
#                    ~3-4x slower than 'none' due to per-layer H2D copies.
#   'group_offload' - CPU offload + leaf-level streaming for the 8B LM (~8 GB; works on 8 GB GPUs).
#                    ~10-20x slower (LM weights paged one layer at a time via CUDA streams).
def _get_repo_dir():
    """Find the local snapshot dir for the MiniMax-Music3 checkpoint."""
    global REPO_DIR
    if REPO_DIR is not None:
        return REPO_DIR
    cache_root = Path(_cache_root_str)
    # HuggingFace hub layout: cache_root/hub/models--MiniMaxAI--MiniMax-Music3/snapshots/<commit>/
    hub_root = cache_root / 'hub' / 'models--MiniMaxAI--MiniMax-Music3' / 'snapshots'
    if hub_root.exists():
        snaps = sorted(hub_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
        if snaps:
            REPO_DIR = str(snaps[0])
            return REPO_DIR
    # Fallback: maybe STEP 2 used a different cache layout
    if (cache_root / 'modular_model_index.json').exists():
        REPO_DIR = str(cache_root)
        return REPO_DIR
    raise FileNotFoundError(
        f'MiniMax-Music3 checkpoint not found at {cache_root}. '
        'Re-run STEP 2 to download it.')
def _detect_vram_gb():
    if not torch.cuda.is_available():
        return 0
    return torch.cuda.get_device_properties(0).total_memory / 1024**3
def _pick_mode():
    """Resolve the form-widget mode + auto-fallback based on detected VRAM."""
    mode = OFFLOAD_MODE
    if mode == 'auto':
        vram = _detect_vram_gb()
        if vram >= 23:
            mode = 'none'
        elif vram >= 15:
            mode = 'cpu_offload'
        else:
            mode = 'group_offload'
        print(f'  Auto-detected {vram:.1f} GB VRAM -> mode={mode}')
    return mode
def _load_pipe(force_reload=False):
    """Lazy pipe loader. Caches the pipe globally so subsequent calls don't reload."""
    global PIPE, PIPE_MODE
    if PIPE is not None and not force_reload:
        return PIPE, PIPE_MODE
    from diffusers import ModularPipeline
    mode = _pick_mode()
    print(f'\nLoading MiniMaxMusic3ModularPipeline (mode={mode}) ...')
    t0 = time.time()
    repo_dir = _get_repo_dir()
    print(f'  Repo dir: {repo_dir}')
    # Load config + components. The modular pipeline is registered
    # via modular_model_index.json which tells diffusers which 7 components
    # (tokenizer, language_model, condition_encoder, rvq_depth_decoder,
    # transformer, vocoder, scheduler) to load and how to wire them.
    pipe = ModularPipeline.from_pretrained(repo_dir)
    pipe.load_components(dtype=torch.bfloat16)
    # Apply the requested offload mode. The pattern is:
    #   1. Wrap the pipeline in a ComponentsManager (needed for any offloading)
    #   2. enable_auto_cpu_offload() for cpu_offload and group_offload modes
    #   3. apply_group_offloading() on the language model for group_offload mode
    if mode == 'none':
        pipe.to('cuda')
        print('  All on cuda (full bf16) - ~24 GB VRAM')
    elif mode in ('cpu_offload', 'group_offload'):
        from diffusers import ComponentsManager
        manager = ComponentsManager()
        manager.enable_auto_cpu_offload(device='cuda')
        # Re-bind the pipe through the manager (re-runs component loading
        # with the offload hooks in place).
        pipe = ModularPipeline.from_pretrained(repo_dir, components_manager=manager)
        pipe.load_components(dtype=torch.bfloat16)
        print('  CPU offload enabled (~16 GB VRAM peak)')
        if mode == 'group_offload':
            from diffusers.hooks.group_offloading import apply_group_offloading
            apply_group_offloading(
                pipe.language_model,
                onload_device=torch.device('cuda'),
                offload_type='leaf_level',
                use_stream=True,
            )
            print('  Group offload applied to language_model (~8 GB VRAM peak)')
    elapsed = time.time() - t0
    print(f'  Pipe loaded in {elapsed:.1f}s')
    print(f'  Sampling rate: {pipe.sampling_rate} Hz, frame rate: {pipe.frame_rate} Hz')
    print(f'  Latent hop length: {pipe.latent_hop_length} samples/frame')
    print(f'  Latent channels: {pipe.num_channels_latents}')
    PIPE = pipe
    PIPE_MODE = mode
    return pipe, mode
def _save_wav(audio_np, sample_rate, out_path):
    """Save a (channels, samples) or (batch, channels, samples) float array as 16-bit PCM WAV."""
    if audio_np.ndim == 3:
        audio_np = audio_np[0]  # drop batch dim
    # Clip to valid range then convert to int16 (the standard WAV format
    # supported by every audio editor).
    audio_np = np.clip(audio_np, -1.0, 1.0)
    audio_int16 = (audio_np.T * 32767.0).astype(np.int16)  # (samples, channels)
    scipy_wav.write(str(out_path), sample_rate, audio_int16)
    return out_path
def _slug(text, max_len=40):
    import re
    s = re.sub(r'[^a-zA-Z0-9_\\-]+', '-', text or '').strip('-')
    return s[:max_len] or 'song'
def generate_song(prompt, lyrics, audio_duration=60.0, num_inference_steps=30,
                  seed=0, save=True, prefix='song', guidance_scale=None):
    """Generate one song and save it as a WAV file.
    Args:
        prompt: music description (genre / mood / vocals / instrumentation / arrangement).
            Free-form text or structured 'Global Metadata / Vocal Details / Arrangement'.
        lyrics: lyrics with structure tags [verse]/[chorus]/[instrumental]/...
                Each tag must be on its own line. Use [instrumental] for no vocals.
        audio_duration: seconds (1.0-300.0). The model may stop earlier via
            the AUDIO_END token. Capped at 9000 acoustic frames (~6 min).
        num_inference_steps: flow-matching Euler steps per chunk (default 30, range 10-100).
            Lower = faster but lower quality. 30 is the official default.
        seed: 0 = random. Otherwise deterministic torch.Generator seed.
        save: write WAV file to OUT_DIR.
        prefix: filename prefix.
        guidance_scale: optional float (default 1.7 from the checkpoint's config).
            Higher = more prompt-adherent but more artifacts; 1.0 = unconditional.
            Only applied to the flow-matching stage. The LM autoregressive stage
            uses its own internal AR_CFG_SCALE which is not user-tunable.
    Returns: dict with keys 'wav_path', 'duration_s', 'sample_rate', 'seed_used', 'mode'.
    """
    pipe, mode = _load_pipe()
    print(f'\nGenerating (mode={mode}, duration={audio_duration}s, steps={num_inference_steps}) ...')
    t0 = time.time()
    generator = None
    seed_used = seed
    if seed <= 0:
        seed_used = random.randint(1, 2**31 - 1)
    generator = torch.Generator(device='cpu').manual_seed(seed_used)

    # Apply guidance_scale to the flow-matching stage's CFG guider.
    # (The LM autoregressive stage uses its own internal AR_CFG_SCALE which
    # is not user-tunable.) Modifying pipe.guider.config.guidance_scale here
    # is the supported way to change CFG strength on a ModularPipeline.
    if guidance_scale is not None and hasattr(pipe, 'guider') and pipe.guider is not None:
        try:
            pipe.guider.config.guidance_scale = float(guidance_scale)
        except Exception as _e:
            print(f'  [WARN] could not set guidance_scale: {_e}')
    
    outputs = pipe(
        prompt=prompt,
        lyrics=lyrics,
        audio_duration=float(audio_duration),
        num_inference_steps=int(num_inference_steps),
        generator=generator,
        output_type='np',
    )
    # Pipeline output: outputs.audios is the standard attribute; fall back
    # to tuple indexing for older diffusers versions.
    audio_np = outputs.audios if hasattr(outputs, 'audios') else outputs[0]
    sr = pipe.sampling_rate
    elapsed = time.time() - t0
    duration_s = audio_np.shape[-1] / sr
    print(f'  Generated {duration_s:.1f}s of audio in {elapsed:.1f}s ({elapsed/duration_s:.1f}x realtime)')
    result = {
        'duration_s': duration_s,
        'sample_rate': sr,
        'seed_used': seed_used,
        'mode': mode,
        'wav_path': None,
    }
    if save:
        slug = _slug(prompt[:30])
        ts = int(time.time())
        out_path = OUT_DIR / f'{prefix}_{slug}_d{int(audio_duration)}s_s{num_inference_steps}_seed{seed_used}_{ts}.wav'
        _save_wav(audio_np, sr, out_path)
        result['wav_path'] = str(out_path)
        print(f'  Saved: {out_path}')
    return result
# Expose to builtins so other cells (STEP 4/6/7) can use generate_song
# without re-importing. The 'MiniMax_MUSIC3_' prefix avoids name collisions
# with other notebooks that might run in the same Python process.
builtins._generate_song = generate_song
builtins._load_pipe = _load_pipe
builtins._save_wav = _save_wav
builtins.MiniMax_MUSIC3_OUT_DIR = OUT_DIR
builtins.MiniMax_MUSIC3_REPO_DIR = lambda: _get_repo_dir()
print('STEP 3 ready. Pipe loads lazily on first generate_song() call.')
print('Use OFFLOAD_MODE form widget to choose none / cpu_offload / group_offload.')
print('Run STEP 4 (Gradio), STEP 6 (quick test), or STEP 7 (batch) next.')


In [ ]:
#@title STEP 4 — (Optional) Gradio UI for MiniMax-Music3
"""
Launches a Gradio app with structured caption fields + lyrics editor + audio player.
The pipe loads lazily on the first Generate click (so the cell returns immediately).
The UI mirrors the official MiniMax Music 3 Space at https://huggingface.co/spaces/MiniMaxAI/MiniMax-Music3
but without ZeroGPU / aoti_load_from_package_dir (those are HF-Spaces-only).
The lyrics field is a multi-line textbox - one structure tag per line, e.g.:
    [verse]
    Walking through the city lights
    Every shadow tells a story
    [chorus]
    We are the night, we are the light
For instrumental music, just put `[instrumental]` on the first line.
"""
import os, sys, time, json, random, builtins
from pathlib import Path
import gradio as gr
import numpy as np
import torch
# Pull generate_song from builtins (exposed by STEP 3)
_generate_song = builtins._generate_song
OUT_DIR = builtins.MiniMax_MUSIC3_OUT_DIR
DEFAULT_CAPTION = (
    'Global Metadata -\n'
    '  Genre: Synthwave\n'
    '  BPM: 100\n'
    '  Key: A minor\n'
    '  Mood: Nostalgic, cinematic\n'
    'Vocal Details -\n'
    '  Gender: Male\n'
    '  Timbre: Warm baritone\n'
    '  Vocal style: Soft, breathy, slightly reverbed\n'
    'Arrangement -\n'
    '  Intro: Soft synth pad + reverb swell, no drums\n'
    '  Verse: Kick + bass drop in, arpeggiated synth carries the rhythm\n'
    '  Chorus: Full band, lead synth melody on top, layered pads\n'
    '  Bridge: Strip back to vocals + piano, builds back to final chorus'
)

DEFAULT_LYRICS = (
    '[verse]\n'
    'Lights are flickering on the boulevard tonight\n'
    'Neon signs reflect in puddles from the afternoon rain\n'
    '[chorus]\n'
    'We chase the shadows down the avenue\n'
    'Finding pieces of ourselves we never knew\n'
    '[verse]\n'
    'Strangers pass like ghosts in colored light\n'
    'Every face a story we will never write\n'
    '[chorus]\n'
    'We chase the shadows down the avenue\n'
    'Finding pieces of ourselves we never knew\n'
    '[outro]\n'
    '(instrumental fade)'
)
def _run(prompt, lyrics, duration, steps, seed, randomize_seed, cfg_scale, save_file, prefix, request: gr.Request):
    if randomize_seed or seed == 0:
        seed = random.randint(1, 2**31 - 1)
    try:
        result = _generate_song(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=float(duration),
            num_inference_steps=int(steps),
            seed=int(seed),
            prefix=prefix if prefix.strip() else 'gradio',
            save=save_file,
            guidance_scale=cfg_scale,
        )
        return (result['wav_path'], result['seed_used'], result['duration_s'])
    except Exception as e:
        raise gr.Error(f'{type(e).__name__}: {e}')
with gr.Blocks(title='MiniMax-Music3', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '## Welcome to MiniMax-Music3\n\n'
        'Enter a **structured caption** (or free-form description) and **tagged lyrics** above.\n'
        'Structure tags like `[verse]`, `[chorus]`, `[instrumental]` must each be on their own line.\n\n'
        'The pipe loads lazily on the first Generate click (30-60s init). Each song takes 2-15 min.\n\n'
        'Use the **Randomize seed** checkbox to try different generations of the same prompt.\n\n'
        'Default caption + lyrics below are a lo-fi / indie-pop example - replace with your own.'
    )
    gr.Markdown('# 🎵 MiniMax-Music3\nLyrics- and caption-conditioned music generation. 44.1 kHz stereo.')
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(value=DEFAULT_CAPTION, label='Structured Caption',
                                lines=10, info='Genre/mood/vocals/instrumentation/arrangement. Free-form text is fine too.')
            lyrics = gr.Textbox(value=DEFAULT_LYRICS, label='Lyrics (one structure tag per line)',
                                lines=12, info='[verse]/[chorus]/[bridge]/[instrumental]/[solo]/[intro]/[outro] etc.')
        with gr.Column(scale=1):
            duration = gr.Slider(minimum=10, maximum=300, value=60, step=5, label='Duration (seconds)')
            steps = gr.Slider(minimum=10, maximum=100, value=30, step=1, label='Inference steps (per chunk)')
            seed = gr.Number(value=0, label='Seed (0 = random)', precision=0)
            randomize_seed = gr.Checkbox(value=True, label='Randomize seed')
            cfg_scale = gr.Slider(minimum=1.0, maximum=5.0, value=1.7, step=0.1,
                                   label='CFG scale (flow-matching guidance)',
                                   info='Lower=more creative, higher=more prompt-adherent')
            save_file = gr.Checkbox(value=True, label='Save WAV to Drive')
            prefix = gr.Textbox(value='gradio', label='Filename prefix',
                                 info='Used as the saved file name prefix')
            btn = gr.Button('Generate', variant='primary')
            out_audio = gr.Audio(label='Generated song', type='filepath')
            out_seed = gr.Textbox(label='Seed used', interactive=False)
            out_duration = gr.Textbox(label='Duration (s)', interactive=False)
    def _show_welcome():
        return '**MiniMax-Music3 ready.** Click Generate to start. The pipe loads on first use (~30-60s).'
    demo.load(_show_welcome, inputs=None, outputs=None)
    btn.click(
        fn=_run,
        inputs=[prompt, lyrics, duration, steps, seed, randomize_seed, cfg_scale, save_file, prefix],
        outputs=[out_audio, out_seed, out_duration],
        api_name='generate',
    )
# Cell 5 launch — use share=False for Colab (we don't need a public URL).
# concurrency_limit=1 since each gen takes 2-15 minutes and the pipe is a single global.
# clear_output() keeps the cell from filling up with status prints.
from IPython.display import display, clear_output
clear_output()
demo.queue(concurrency_limit=1).launch(share=False, inline=False, prevent_thread_lock=True, quiet=True)
display(demo)
print('STEP 4 launched. The Gradio UI is interactive in the output above.')
print('Each generation takes 2-15 min depending on GPU + duration. The pipe is shared, so concurrent requests are serialized.')

In [ ]:
#@title STEP 5 — Keep Colab alive + session summary
"""
Prevents Colab from disconnecting by polling a heartbeat (any heavy import works).
Also prints a summary of the current session state.
"""
import time, json, os
from pathlib import Path
# Session summary
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        vram_gb = p.total_memory / 1024**3
        print(f'GPU: {p.name} ({vram_gb:.1f} GB)')
    else:
        print('GPU: none')
except Exception:
    print('GPU: unavailable')
try:
    import diffusers
    print(f'diffusers: {diffusers.__version__}')
except Exception:
    print('diffusers: not installed')
try:
    repo_dir = builtins.MiniMax_MUSIC3_REPO_DIR()
    print(f'Checkpoint: {repo_dir}')
    # Total size
    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(repo_dir) for f in files
    )
    print(f'  Total on disk: {total / 1024**3:.2f} GB')
except Exception:
    print('Checkpoint: not found (re-run STEP 2)')
try:
    out_dir = builtins.MiniMax_MUSIC3_OUT_DIR
    wavs = sorted(out_dir.glob('*.wav'))
    print(f'Output dir : {out_dir}')
    print(f'  songs: {len(wavs)}')
    total_audio_s = 0
    for w in wavs[-5:]:
        import scipy.io.wavfile as _wav
        sr, audio = _wav.read(str(w))
        secs = audio.shape[0] / sr
        total_audio_s += secs
        print(f'    {secs:.1f}s  {w.name}')
except Exception as e:
    print(f'Output dir: {e}')
# Heartbeat loop (re-run STEP 5 to interrupt)
print('\nHeartbeat loop (Ctrl+C or Interrupt to stop):')
try:
    while True:
        time.sleep(60)
        print(f'  ... alive at {time.strftime("%H:%M:%S")}')
except KeyboardInterrupt:
    print('Heartbeat stopped.')

In [ ]:
#@title STEP 6 — Quick test (single song generation)
"""
Generate one song with form-widget inputs and immediately play it. Useful for
verifying the pipeline works after install + download.
The pipe is loaded lazily on first call (STEP 3 init). Expect ~30-60s
initialization plus the generation time (~2-7 min for 60s of audio).
"""
import time, json, random, builtins
from pathlib import Path
from IPython.display import display, Audio, FileLink
# Form widget for the prompt (structured caption or free-form music description).
# Edit in the form field to the left of the cell. Default: a lo-fi indie-pop example.
# (The text is stored as a single-line string; the \n escapes are real newlines at runtime.)
PROMPT = 'Global Metadata -\n\n  Genre: Lo-fi hip hop\n\n  BPM: 85\n\n  Key: C major\n\n  Mood: Relaxed, study-friendly, warm\n\nVocal Details -\n\n  Gender: Female\n\n  Timbre: Soft alto with gentle vibrato\n\n  Vocal style: Whispered, intimate\n\nArrangement -\n\n  Intro: 4 bars of vinyl crackle + soft piano chord\n\n  Verse: Mellow kick + sub bass, jazzy Rhodes chords\n\n  Chorus: Add brushed snare + warm string pad, vocal doubles\n\n  Bridge: Strip to piano + vocal, then build back to final chorus'  #@param {type:'string'}

# Form widget for the lyrics. Each structure tag ([verse], [chorus], [instrumental], ...)
# must be on its own line in the text below. Use [instrumental] for instrumental music.
# (The text is stored as a single-line string; the \n escapes are real newlines at runtime.)
LYRICS = '[intro]\n\n(instrumental)\n\n[verse]\n\nCoffee steam is curling up against the windowpane\n\nNotes scattered on the desk like fallen leaves again\n\n[chorus]\n\nLet the morning keep on moving slow\n\nThere is nowhere else I need to go\n\n[verse]\n\nPages turning like the hands upon a quieter clock\n\nEvery shadow dancing where the sun begins to walk\n\n[chorus]\n\nLet the morning keep on moving slow\n\nThere is nowhere else I need to go\n\n[outro]\n\n(instrumental fade)'  #@param {type:'string'}
DURATION = 60  #@param {type:'slider', min:10, max:300, step:5}
STEPS = 30  #@param {type:'slider', min:10, max:100, step:1}
SEED = 0  #@param {type:'integer'}
RANDOMIZE_SEED = True  #@param {type:'boolean'}
# If True, ignores SEED and picks a fresh random seed each run.
# Useful for exploring different generations of the same prompt.
CFG_SCALE = 1.7  #@param {type:'slider', min:1.0, max:5.0, step:0.1}
# Flow-matching CFG scale (default 1.7 from checkpoint config).
# Lower (1.0) = more creative/loose; higher (3-5) = more prompt-adherent.
SAVE_FILE = True  #@param {type:'boolean'}
# If False, song is generated but NOT written to disk (preview only).
PREFIX = 'quicktest'  #@param {type:'string'}
# Filename prefix for the saved WAV (only used if SAVE_FILE=True).


print(f'  Prompt  : {PROMPT[:60]}...')
print(f'  Lyrics  : {LYRICS.splitlines()[0] if LYRICS else ""}...')
print(f'  Duration: {DURATION}s, steps={STEPS}, seed={SEED} (0 = random)')
print(f'  CFG: {CFG_SCALE}, randomize={RANDOMIZE_SEED}, save={SAVE_FILE}, prefix={PREFIX!r}')
print()

result = builtins._generate_song(
    prompt=PROMPT,
    lyrics=LYRICS,
    audio_duration=DURATION,
    num_inference_steps=STEPS,
    seed=SEED if not RANDOMIZE_SEED else 0,
    save=SAVE_FILE,
    prefix=PREFIX if PREFIX.strip() else 'quicktest',
    guidance_scale=CFG_SCALE,
)

print(f'  Done: {result["duration_s"]:.1f}s of audio, seed={result["seed_used"]}')
print(f'  File: {result["wav_path"]}')
print()
display(Audio(result['wav_path']))
try:
    display(FileLink(result['wav_path']))
except Exception:
    pass


In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list
"""
Reads a JSON file containing a list of songs and generates them sequentially.
The pipe is loaded once and reused across all entries. Per-entry skip-existing
with hash-based change detection (same scheme as MiniMax-H3 + LTX-2.5 notebooks).
JSON format (a list of objects) -
```json
[
  {
    "prompt": "Global Metadata: Genre: Synthwave, BPM: 110, ...",
    "lyrics": "[verse]\nLyrics line 1\n[chorus]\nLyrics line 2",
    "duration": 60,
    "steps": 30,
    "seed": 42
  },
  {
    "prompt": "..."
    "lyrics": "[instrumental]"
    "duration": 30,
    "seed": 123
  }
]
```
Fields (only `prompt` and `lyrics` are required; rest override the form-widget defaults):
  - prompt:    (required) structured caption OR free-form music description.
  - lyrics:    (required) lyrics with [verse]/[chorus]/[instrumental] tags, one per line.
  - duration:  (optional, default 60) seconds (1-300, capped at 9000 latent frames ~ 6 min).
  - steps:     (optional, default 30) flow-matching Euler steps per chunk (10-100).
  - seed:      (optional, default 0 = random) int seed for reproducibility.
  - cfg_scale: (optional, default 1.7) flow-matching CFG scale (1.0-5.0).
The file at BATCH_JSON_PATH is created with a starter template if it does not exist.
"""
import os, sys, time, json, random, hashlib, builtins
from pathlib import Path
BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/MiniMax-Music3/batch_songs.json'  #@param {type:'string'}
DEFAULT_DURATION = 60  #@param {type:'slider', min:10, max:300, step:5}
DEFAULT_STEPS    = 30  #@param {type:'slider', min:10, max:100, step:1}
DEFAULT_CFG_SCALE = 1.7  #@param {type:'slider', min:1.0, max:5.0, step:0.1}
# Default CFG scale for entries that don't specify it.
SKIP_EXISTING = True  #@param {type:'boolean'}
RESUME_FROM_LOG = True  #@param {type:'boolean'}
batch_path = Path(BATCH_JSON_PATH)
if not batch_path.exists():
    batch_path.parent.mkdir(parents=True, exist_ok=True)
    starter = [
        {
            'prompt': 'Global Metadata: Genre: Ambient electronic, BPM: 90, Key: D minor, Mood: Reflective, introspective.\nVocal Details: No vocals (instrumental).\nArrangement: Slow build with pads, sparse piano, no percussion.',
            'lyrics': '[instrumental]\n(soft pad swells)\n(gentle piano motif repeats)\n(careful layering of textures)\n(gentle fade)',
            'duration': 60,
            'steps': 30,
            'seed': 0
        },
        {
            'prompt': 'Global Metadata: Genre: Indie folk, BPM: 95, Key: G major, Mood: Uplifting, hopeful.\nVocal Details: Gender: Male, Timbre: Warm tenor, Vocal style: Bright, clear.\nArrangement: Acoustic guitar arpeggios, light brushed drums, layered harmonies on chorus.',
            'lyrics': '[verse]\nWaking up to morning sun through an open window\nThe world outside is waking too\n[chorus]\nEvery day a chance to start again\nEvery step a door that opens from within\n[verse]\nCoffee steam and quiet thoughts in a kitchen chair\nThe simple things that bring us there\n[chorus]\nEvery day a chance to start again\nEvery step a door that opens from within',
            'duration': 60,
            'steps': 30,
            'seed': 0
        }
    ]
    batch_path.write_text(json.dumps(starter, indent=2))
    print(f'  Wrote starter batch: {batch_path}')
    print('  Edit it (add prompt, lyrics, duration, steps, seed), then re-run STEP 7.')
else:
    print(f'  Using existing batch JSON: {batch_path}')
scenes = json.loads(batch_path.read_text())
if not isinstance(scenes, list):
    raise SystemExit(f'Batch JSON must be a list, got {type(scenes).__name__}')
print(f'\n  Scenes in batch: {len(scenes)}')
# Resume log (one JSON line per scene, append-only)
_log_path = builtins.MiniMax_MUSIC3_OUT_DIR / 'batch_songs.log.jsonl'
_completed = set()
if RESUME_FROM_LOG and _log_path.exists():
    for line in _log_path.read_text().splitlines():
        try:
            entry = json.loads(line)
            if entry.get('status') == 'ok':
                _completed.add(entry['index'])
        except Exception:
            pass
    print(f'  Resume log: {len(_completed)} already-completed')
def _song_prompt_hash(prompt, lyrics, duration, steps):
    h = hashlib.sha256()
    h.update(prompt.encode('utf-8'))
    h.update(b'\x00')
    h.update(lyrics.encode('utf-8'))
    h.update(b'\x00')
    h.update(str(duration).encode())
    h.update(b'\x00')
    h.update(str(steps).encode())
    h.update(b'\x00')
    h.update(str(cfg_scale).encode())
    return h.hexdigest()[:16]
results = []
total_start = time.time()
for i, sc in enumerate(scenes):
    if i in _completed:
        print(f'  [{i+1}/{len(scenes)}] SKIP (resume log)')
        results.append(None)
        continue
    prompt = (sc.get('prompt') or '').strip()
    lyrics = (sc.get('lyrics') or '').strip()
    if not prompt:
        print(f'  [{i+1}/{len(scenes)}] SKIP: empty prompt')
        results.append(None)
        continue
    if not lyrics:
        print(f'  [{i+1}/{len(scenes)}] SKIP: empty lyrics')
        results.append(None)
        continue
    duration = int(sc.get('duration', DEFAULT_DURATION))
    steps = int(sc.get('steps', DEFAULT_STEPS))
    seed = int(sc.get('seed', 0))
    cfg_scale = float(sc.get('cfg_scale', DEFAULT_CFG_SCALE))
    hash_id = _song_prompt_hash(prompt, lyrics, duration, steps)
    existing = sorted(builtins.MiniMax_MUSIC3_OUT_DIR.glob(f'*_seed*_{hash_id[:8]}_*.wav'))
    if SKIP_EXISTING and existing:
        print(f'  [{i+1}/{len(scenes)}] SKIP (hash match): {existing[0].name}')
        results.append(str(existing[0]))
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'ok', 'path': str(existing[0]),
                                'hash': hash_id, 'skipped': True}) + chr(10))
        continue
    print(f'\n  [{i+1}/{len(scenes)}] d={duration}s steps={steps} seed={seed} hash={hash_id[:8]}')
    print(f'    prompt: {prompt[:60]}...')
    print(f'    lyrics: {lyrics[:60].splitlines()[0] if lyrics else ""}...')
    try:
        t0 = time.time()
        result = builtins._generate_song(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=duration,
            num_inference_steps=steps,
            seed=seed,
            save=True,
            prefix=f'batch{i+1:03d}',
            guidance_scale=cfg_scale,
        )
        elapsed = time.time() - t0
        print(f'    -> {elapsed:.0f}s, {result["duration_s"]:.1f}s audio')
        results.append(result['wav_path'])
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'ok',
                                'path': result['wav_path'],
                                'duration_s': result['duration_s'],
                                'elapsed_s': elapsed,
                                'seed': result['seed_used'],
                                'hash': hash_id}) + chr(10))
    except Exception as e:
        print(f'    FAIL: {type(e).__name__}: {e}')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'fail',
                                'error': f'{type(e).__name__}: {e}'}) + chr(10))
# Batch summary
total = time.time() - total_start
ok_count = sum(1 for r in results if r is not None)
fail_count = sum(1 for r in results if r is None and i < len(scenes))
print('\n' + '='*72)
print(f'  BATCH SUMMARY')
print('='*72)
print(f'  Scenes:  {ok_count} succeeded, {len(scenes) - ok_count} failed/skipped, {len(scenes)} total')
print(f'  Total:   {_fmt_seconds(total)}')
print(f'  Log:     {_log_path}')
print()
print('  Output files:')
for r in results:
    if r:
        print(f'    {r}')
print('='*72)
def _fmt_seconds(seconds):
    s = int(seconds); h, rem = divmod(s, 3600); m, ss = divmod(rem, 60)
    return f'{h}:{m:02d}:{ss:02d}' if h else f'{m:02d}:{ss:02d}'

In [ ]:
#@title STEP 9 — Mux generated song with a video file (music video output)
"""
Combine a generated song (WAV) with a video file (mp4/mov) into a single
output video. Useful for music video workflows where you generate:
  - Music: here in this notebook (MiniMax-Music3)
  - Video: in the LTX-2-5_ComfyUI_Colab notebook (or anywhere else)
Uses ffmpeg (bundled with Colab, no install needed). Audio is encoded as AAC
128 kbps, video is copied without re-encoding (preserves quality).
Three modes:
  - Replace: strip the video audio, replace with the song (most common for music videos)
  - Mix: mix the song with the video audio (e.g. for ambient scenes)
  - Concat: loop the song to match the video duration, then mux (the video is left untouched)
Input widgets let you pick the video + song. The output goes to the same
OUT_DIR as the generated songs, with a clear filename pattern.
"""
import os, sys, time, json, subprocess, shutil, pathlib, builtins
from pathlib import Path
from IPython.display import display, FileLink
OUT_DIR = builtins.MiniMax_MUSIC3_OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
# --- Form widgets ---
VIDEO_PATH = '/content/drive/MyDrive/AEI_3D_Out/LTX-Video-2.5/your_video.mp4'  #@param {type:'string'}
# Find the most recent WAV if not specified
_default_song = max(OUT_DIR.glob('*.wav'), key=lambda p: p.stat().st_mtime, default=None)
SONG_PATH = str(_default_song) if _default_song else str(OUT_DIR / 'your_song.wav')  #@param {type:'string'}
MUX_MODE = 'Replace'  #@param ['Replace', 'Mix', 'Concat']
MIX_VOLUME = 0.3  #@param {type:'slider', min:0.0, max:2.0, step:0.05}
AUDIO_BITRATE = '128k'  #@param ['96k', '128k', '192k', '256k', '320k']
# --- Validate ---
video_p = Path(VIDEO_PATH)
song_p = Path(SONG_PATH)
if not video_p.exists():
    raise SystemExit(f'Video not found: {video_p}')
if not song_p.exists():
    raise SystemExit(f'Song not found: {song_p}')
# Check ffmpeg is available
if not shutil.which('ffmpeg'):
    print('  ffmpeg not found - installing via apt...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=False)
    if not shutil.which('ffmpeg'):
        raise SystemExit('ffmpeg install failed. Try: !apt-get install ffmpeg')
ffmpeg_path = shutil.which('ffmpeg')
ffprobe_path = shutil.which('ffprobe') or ffmpeg_path.replace('ffmpeg', 'ffprobe')
# Probe the video for duration + streams
print(f'  Video: {video_p.name}')
print(f'  Song:  {song_p.name}')
print(f'  Mode:  {MUX_MODE}')
_probe = subprocess.run(
    [ffprobe_path, '-v', 'error', '-show_entries', 'format=duration,size',
     '-show_entries', 'stream=codec_type,codec_name,sample_rate,channels',
     '-of', 'default=nw=1', str(video_p)],
    capture_output=True, text=True, timeout=60,
)
video_dur_s = 0
for line in _probe.stdout.splitlines():
    if line.startswith('duration='):
        try: video_dur_s = float(line.split('=')[1])
        except: pass
print(f'  Video duration: {video_dur_s:.1f}s')
# Probe the song
_probe = subprocess.run(
    [ffprobe_path, '-v', 'error', '-show_entries', 'format=duration',
     '-of', 'default=nw=1', str(song_p)],
    capture_output=True, text=True, timeout=60,
)
song_dur_s = 0
for line in _probe.stdout.splitlines():
    if line.startswith('duration='):
        try: song_dur_s = float(line.split('=')[1])
        except: pass
print(f'  Song duration:  {song_dur_s:.1f}s')
# Build output path
ts = int(time.time())
out_path = OUT_DIR / f'mux_{MUX_MODE.lower()}_{video_p.stem}_{song_p.stem}_{ts}.mp4'
# Build ffmpeg command based on mode
if MUX_MODE == 'Replace':
    # Strip video's audio, use song as the only audio. Loop the song if
    # shorter than the video (e.g. 60s song + 5s video = 5s of song, no loop needed).
    cmd = [
        ffmpeg_path, '-y',
        '-i', str(video_p),
        '-i', str(song_p),
        '-map', '0:v:0',  # video from video_p
        '-map', '1:a:0',  # audio from song_p
        '-c:v', 'copy',   # don't re-encode video
        f'-b:a', AUDIO_BITRATE,  # audio bitrate
        '-shortest',  # trim to shorter of the two
        '-movflags', '+faststart',  # web-friendly MP4
        str(out_path),
    ]
elif MUX_MODE == 'Mix':
    # Mix the song with the video's existing audio at MIX_VOLUME.
    cmd = [
        ffmpeg_path, '-y',
        '-i', str(video_p),
        '-i', str(song_p),
        '-filter_complex',
        f'[1:a]volume={MIX_VOLUME}[song];[0:a][song]amix=inputs=2:duration=shortest[aout]',
        '-map', '0:v:0',
        '-map', '[aout]',
        '-c:v', 'copy',
        f'-b:a', AUDIO_BITRATE,
        '-shortest',
        '-movflags', '+faststart',
        str(out_path),
    ]
else:  # Concat
    # Loop the song (or trim it) so it matches the video duration.
    # amovie with -stream_loop -1 loops indefinitely; -shortest then trims
    # at the video's end.
    cmd = [
        ffmpeg_path, '-y',
        '-i', str(video_p),
        '-stream_loop', '-1', '-i', str(song_p),
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        f'-b:a', AUDIO_BITRATE,
        '-shortest',
        '-movflags', '+faststart',
        str(out_path),
    ]
print(f'  ffmpeg: starting...')
print(f'  Output: {out_path}')
print(f'  Command: {" ".join(cmd[:6])}...')
print()
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0
if result.returncode != 0:
    print(f'  [FAIL] ffmpeg error (rc={result.returncode}):')
    print(result.stderr[-1500:])
    raise SystemExit(f'Muxing failed. See stderr above.')
print(f'  Done in {elapsed:.1f}s.')
print(f'  Size: {out_path.stat().st_size / 1024 / 1024:.1f} MB')
print()
display(FileLink(str(out_path)))


In [ ]:
#@title STEP 8 — Tail the latest log / errors (debug aid)
"""
Prints the tail of the batch log and any error messages. Useful when a generation
fails or you want to see which scenes completed.
"""
import json
from pathlib import Path
_log_path = builtins.MiniMax_MUSIC3_OUT_DIR / 'batch_songs.log.jsonl'
TAIL_LINES = 30  #@param {type:'slider', min:5, max:200, step:5}
if not _log_path.exists():
    print(f'No log file yet at {_log_path}')
    print('Run STEP 7 to generate songs (creates the log).')
else:
    lines = _log_path.read_text().splitlines()
    tail = lines[-TAIL_LINES:]
    print(f'Tail of {_log_path.name} ({len(lines)} total entries):\n')
    for line in tail:
        try:
            entry = json.loads(line)
            ts = entry.get('index', '?')
            st = entry.get('status', '?')
            extras = []
            if 'path' in entry:
                extras.append(f'path={Path(entry["path"]).name}')
            if 'duration_s' in entry:
                extras.append(f'duration={entry["duration_s"]:.1f}s')
            if 'elapsed_s' in entry:
                extras.append(f'elapsed={entry["elapsed_s"]:.0f}s')
            if 'seed' in entry:
                extras.append(f'seed={entry["seed"]}')
            if 'error' in entry:
                extras.append(f'error={entry["error"]}')
            extras_str = ' '.join(extras)
            print(f'  [{ts:>3}] {st:5s} {extras_str}')
        except Exception:
            print(f'  (raw) {line[:200]}')